# 01 Data Audit

Thin review notebook for artifacts produced by `make validate-data` and `make build-cohort`. This notebook inspects saved outputs only; it does not construct cohorts, define causal variables, or run estimators.

In [ ]:
from pathlib import Path
import json
import sys

from IPython.display import Image, display
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from oulad_causal.config import FIGURES_DIR, METADATA_DIR, PROCESSED_DATA_DIR

metadata_dir = METADATA_DIR
processed_dir = PROCESSED_DATA_DIR
figures_dir = FIGURES_DIR
metadata_dir, processed_dir, figures_dir

## Raw Validation Artifacts

In [ ]:
table_shapes = pd.read_csv(metadata_dir / "table_shapes.csv")
schema = pd.read_csv(metadata_dir / "schema_validation.csv")
duplicates = pd.read_csv(metadata_dir / "duplicate_key_summary.csv")
missingness = pd.read_csv(metadata_dir / "missingness_summary.csv")
dates = pd.read_csv(metadata_dir / "date_range_summary.csv")
categories = pd.read_csv(metadata_dir / "category_frequency_summary.csv")

table_shapes

In [ ]:
schema[["table_name", "missing_columns", "extra_columns", "column_order_matches", "is_valid"]]

In [ ]:
ax = table_shapes.sort_values("row_count").plot.barh(x="table_name", y="row_count", legend=False, figsize=(8, 4))
ax.set_xlabel("Rows")
ax.set_ylabel("")
ax.set_title("Raw OULAD Table Sizes")
plt.tight_layout()

In [ ]:
missingness.sort_values("missing_fraction", ascending=False).head(20)

In [ ]:
final_result = categories[(categories["table_name"] == "studentInfo") & (categories["column"] == "final_result")]
ax = final_result.sort_values("count").plot.barh(x="category", y="count", legend=False, figsize=(7, 3))
ax.set_xlabel("Students")
ax.set_ylabel("")
ax.set_title("Raw Final Result Labels")
plt.tight_layout()

In [ ]:
dates

## Analytic Cohort Artifacts

In [ ]:
cohort_path = processed_dir / "oulad_analytic_cohort.parquet"
flow_path = processed_dir / "cohort_flow_table.csv"
summary_path = processed_dir / "cohort_summary.json"

cohort = pd.read_parquet(cohort_path)
cohort_flow = pd.read_csv(flow_path)
cohort_summary = json.loads(summary_path.read_text())

cohort.shape, cohort_summary["cohort_size"]

In [ ]:
cohort_flow

In [ ]:
primary_treatment_columns = [
    "treatment_high_engagement_14d_median",
    "treatment_high_engagement_14d_top_tertile",
    "treatment_high_engagement_14d_top_quartile",
]
pd.DataFrame(
    [
        {
            "treatment": column,
            "nonmissing": int(cohort[column].notna().sum()),
            "treated": int(cohort[column].sum()),
            "prevalence": float(cohort[column].mean()),
        }
        for column in primary_treatment_columns
    ]
)

In [ ]:
cohort[[
    "code_module",
    "code_presentation",
    "id_student",
    "final_result",
    "outcome_success",
    "early_clicks_14d",
    "early_clicks_14d_z",
    "treatment_high_engagement_14d_median",
]].head()

In [ ]:
for figure_name in ["cohort_flow.png", "treatment_prevalence.png"]:
    display(Image(filename=str(figures_dir / figure_name)))